# TVL v2 — 賽前預測引擎

## 與 v1 (02_ml_match_prediction) 的根本差異

| | v1 (賽後解讀) | v2 (賽前預測) |
|---|---|---|
| **特徵** | 當場比賽的 ASR、GP% 等 | 過去 N 場的歷史滾動平均 |
| **Data Leakage** | 嚴重：用答案預測答案 | 已修正：`shift(1)` 排除當場 |
| **驗證策略** | `StratifiedKFold` 隨機切分 | `TimeSeriesSplit` 時間序列切分 |
| **超參數** | 手動設定 | `Optuna` 自動搜索 |
| **實戰價值** | 無（事後諸葛） | 有（賽前可用的資訊） |

### 流程
1. 資料準備與勝負標籤推論（沿用 v1）
2. **任務 1**：滾動歷史特徵 (Rolling Historical Features)
3. **任務 2**：時間序列交叉驗證 (TimeSeriesSplit)
4. **任務 3**：Optuna 自動調參
5. 最終模型 SHAP 分析與匯出

---
## 0. 環境設定

In [1]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
)

import xgboost as xgb
import optuna
import shap
import joblib

matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# 降低 Optuna 調參過程的 log 雜訊
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

print("環境設定完成。")

環境設定完成。


---
## 1. 資料準備與勝負標籤推論（沿用 v1）

此段邏輯與 `02_ml_match_prediction.ipynb` 相同，濃縮為兩個 cell。

In [2]:
# ── 1.1 讀取 DB → 聚合為球隊單場數據 → 計算單場效率指標 ──────
DB_PATH = Path("../data/db/tvl_database.db")
conn = sqlite3.connect(DB_PATH)

player_stats = pd.read_sql_query("""
    SELECT s.*, p.team_id, p.gender, t.team_name
    FROM player_match_stats s
    JOIN players p ON s.player_id = p.player_id
    JOIN teams   t ON p.team_id = t.team_id AND p.gender = t.gender
""", conn)
conn.close()

team_match = (
    player_stats
    .groupby(["match_date", "team_id", "gender", "team_name", "opponent"])
    .agg(
        total_attack_pts=("attack_points", "sum"),
        total_attack_tot=("attack_total", "sum"),
        total_block_pts=("block_points", "sum"),
        total_serve_pts=("serve_points", "sum"),
        total_serve_tot=("serve_total", "sum"),
        total_rcv_exc=("receive_excellent", "sum"),
        total_rcv_tot=("receive_total", "sum"),
        total_dig_exc=("dig_excellent", "sum"),
        total_dig_tot=("dig_total", "sum"),
        total_points=("total_points", "sum"),
        total_sets=("sets_played", "max"),
    )
    .reset_index()
)

def safe_pct(num, denom):
    return (num / denom * 100) if denom > 0 else 0.0

# 五大單場效率指標（這些是「賽後才知道」的數值，只用來計算滾動歷史）
GAME_STAT_COLS = ["ASR", "GP_pct", "DIG_pct", "BLK_per_set", "ACE_pct"]

team_match["ASR"] = team_match.apply(
    lambda r: safe_pct(r["total_attack_pts"], r["total_attack_tot"]), axis=1)
team_match["GP_pct"] = team_match.apply(
    lambda r: safe_pct(r["total_rcv_exc"], r["total_rcv_tot"]), axis=1)
team_match["DIG_pct"] = team_match.apply(
    lambda r: safe_pct(r["total_dig_exc"], r["total_dig_tot"]), axis=1)
team_match["BLK_per_set"] = team_match.apply(
    lambda r: r["total_block_pts"] / r["total_sets"] if r["total_sets"] > 0 else 0, axis=1)
team_match["ACE_pct"] = team_match.apply(
    lambda r: safe_pct(r["total_serve_pts"], r["total_serve_tot"]), axis=1)

print(f"球隊單場紀錄數：{len(team_match)}")

球隊單場紀錄數：216


In [3]:
# ── 1.2 勝負標籤推論（以團隊總得分比較） ─────────────────────
OPP_SHORT_TO_TEAM = {
    "屏東台電": (1, "M"), "雲林美津濃": (2, "M"), "臺北國北獅": (4, "M"),
    "桃園臺產": (5, "M"), "獅子王": (7, "M"),
    "高雄台電": (4, "F"), "臺北鯨華": (3, "F"),
    "新北中纖": (5, "F"), "義力營造": (7, "F"),
}

pts_lookup = (
    team_match.set_index(["match_date", "team_id", "gender"])["total_points"]
    .to_dict()
)

labels = []
for _, row in team_match.iterrows():
    opp = OPP_SHORT_TO_TEAM.get(row["opponent"])
    if opp is None:
        labels.append(np.nan)
        continue
    opp_pts = pts_lookup.get((row["match_date"], opp[0], opp[1]))
    if opp_pts is None:
        labels.append(np.nan)
    elif row["total_points"] > opp_pts:
        labels.append(1)
    elif row["total_points"] < opp_pts:
        labels.append(0)
    else:
        labels.append(np.nan)  # 平手排除

team_match["win"] = labels
labeled = team_match.dropna(subset=["win"]).copy()
labeled["win"] = labeled["win"].astype(int)

print(f"已標記勝負：{len(labeled)} 筆 (勝 {labeled['win'].sum()} / 敗 {(labeled['win']==0).sum()})")

已標記勝負：212 筆 (勝 106 / 敗 106)


---
## 2. 任務 1：滾動歷史特徵 (Rolling Historical Features)

### 核心觀念：為什麼 v1 的特徵有 Data Leakage？

v1 直接把「當場比賽的 ASR = 45%」當作特徵，去預測「當場比賽是否獲勝」。
但在實戰中，比賽開始前你不知道這場 ASR 會是多少 — 這是「拿考卷答案去預測考試結果」。

**解決方案：** 特徵只能使用「比賽開始前已知的資訊」：
- 該隊過去 N 場比賽的歷史平均表現（滾動平均）
- 該隊近期的連勝/連敗狀態

### 實作注意事項

1. **`shift(1)`**：確保滾動窗口不包含當場比賽
2. **同日比賽排序**：`match_date` 僅有日期無時間，同日若有多場比賽，
   以 `opponent` 名稱作為穩定排序的 tiebreaker（非完美，但可確保排序一致性）
3. **標籤污染風險**：連勝/連敗特徵使用的 proxy label 本身可能有誤差，
   該誤差會透過滾動計算放大 — 這是目前標籤推論方法的固有限制

In [4]:
# ── 2.1 嚴格按 team_id + match_date 排序 ─────────────────────
# 同日多場比賽時，以 opponent 作為穩定 tiebreaker
df = (
    labeled
    .sort_values(["team_id", "gender", "match_date", "opponent"])
    .reset_index(drop=True)
)

# 確認排序結果：每隊的比賽按時間排列
sample_team = df["team_id"].iloc[0]
sample_gender = df["gender"].iloc[0]
sample = df[(df["team_id"] == sample_team) & (df["gender"] == sample_gender)]
print(f"排序驗證 — team_id={sample_team}, gender={sample_gender}:")
print(sample[["match_date", "opponent", "win"]].to_string(index=False))

排序驗證 — team_id=1, gender=M:
match_date opponent  win
2025-11-01    雲林美津濃    0
2025-11-02    臺北國北獅    1
2025-11-08      獅子王    1
2025-11-09     桃園臺產    1
2025-11-15      獅子王    1
2025-11-16     桃園臺產    0
2025-11-29      獅子王    1
2025-11-30    臺北國北獅    1
2025-12-14      獅子王    1
2025-12-20     桃園臺產    1
2025-12-21    雲林美津濃    1
2026-01-03    臺北國北獅    1
2026-01-04    雲林美津濃    1
2026-01-10     桃園臺產    1
2026-01-11    臺北國北獅    1
2026-01-24     桃園臺產    1
2026-01-25     桃園臺產    1
2026-01-31      獅子王    1
2026-02-01    臺北國北獅    1
2026-03-07    雲林美津濃    1
2026-03-08    臺北國北獅    1
2026-03-14      獅子王    1
2026-03-15    雲林美津濃    1


In [5]:
# ── 2.2 計算近 3 場 / 近 5 場滾動平均 ─────────────────────────
# shift(1): 排除當場數據，只看「過去」
# min_periods=1: 開季前幾場歷史不足時，用已有的場次計算

group_keys = ["team_id", "gender"]

for col in GAME_STAT_COLS:
    shifted = df.groupby(group_keys)[col].shift(1)

    df[f"{col}_roll3"] = (
        shifted
        .groupby(df[group_keys].apply(tuple, axis=1))
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )
    df[f"{col}_roll5"] = (
        shifted
        .groupby(df[group_keys].apply(tuple, axis=1))
        .transform(lambda x: x.rolling(5, min_periods=1).mean())
    )

print("滾動特徵計算完成。")
print(f"新增欄位：{[c for c in df.columns if 'roll' in c]}")

滾動特徵計算完成。
新增欄位：['ASR_roll3', 'ASR_roll5', 'GP_pct_roll3', 'GP_pct_roll5', 'DIG_pct_roll3', 'DIG_pct_roll5', 'BLK_per_set_roll3', 'BLK_per_set_roll5', 'ACE_pct_roll3', 'ACE_pct_roll5']


In [6]:
# ── 2.3 計算連勝/連敗狀態 (Win Streak) ────────────────────────
# 正值 = 連勝場數，負值 = 連敗場數，0 = 無歷史（開季首場）

def compute_win_streak(wins: pd.Series) -> list[int]:
    """
    計算每場比賽「開打前」的連勝/連敗狀態。
    使用 shift 概念：第 i 場的 streak 只反映前 i-1 場的結果。
    """
    streaks = []
    current = 0
    for w in wins:
        streaks.append(current)  # 先記錄「賽前」的 streak
        if w == 1:
            current = current + 1 if current > 0 else 1
        else:
            current = current - 1 if current < 0 else -1
    return streaks


df["win_streak"] = (
    df.groupby(group_keys)["win"]
    .transform(compute_win_streak)
)

# 驗證：顯示某隊的 streak 計算是否正確
print("Win Streak 驗證（同一隊的 win vs streak）：")
print(sample_team_check := df[
    (df["team_id"] == sample_team) & (df["gender"] == sample_gender)
][["match_date", "opponent", "win", "win_streak"]].head(10).to_string(index=False))

Win Streak 驗證（同一隊的 win vs streak）：
match_date opponent  win  win_streak
2025-11-01    雲林美津濃    0           0
2025-11-02    臺北國北獅    1          -1
2025-11-08      獅子王    1           1
2025-11-09     桃園臺產    1           2
2025-11-15      獅子王    1           3
2025-11-16     桃園臺產    0           4
2025-11-29      獅子王    1          -1
2025-11-30    臺北國北獅    1           1
2025-12-14      獅子王    1           2
2025-12-20     桃園臺產    1           3


In [7]:
# ── 2.4 清理：移除首場比賽（無任何歷史特徵）──────────────────
ROLLING_FEATURES = (
    [f"{c}_roll3" for c in GAME_STAT_COLS]
    + [f"{c}_roll5" for c in GAME_STAT_COLS]
    + ["win_streak"]
)

# 首場比賽的 roll 欄位為 NaN（shift(1) 的結果），移除這些列
n_before = len(df)
df_clean = df.dropna(subset=[f"{GAME_STAT_COLS[0]}_roll3"]).copy()
n_dropped = n_before - len(df_clean)

print(f"移除各隊首場（無歷史）紀錄：{n_dropped} 筆")
print(f"可用訓練樣本數：{len(df_clean)} 筆")
print(f"\n滾動特徵矩陣預覽：")
df_clean[ROLLING_FEATURES].describe().round(2)

移除各隊首場（無歷史）紀錄：9 筆
可用訓練樣本數：203 筆

滾動特徵矩陣預覽：


,ASR_roll3,GP_pct_roll3,DIG_pct_roll3,BLK_per_set_roll3,ACE_pct_roll3,ASR_roll5,GP_pct_roll5,DIG_pct_roll5,BLK_per_set_roll5,ACE_pct_roll5,win_streak
count,203.00,203.00,203.00,203.00,203.00,203.00,203.00,203.00,203.00,203.00,203.00
mean,41.77,50.40,31.69,1.81,3.92,41.80,50.30,31.77,1.80,3.88,0.34
std,7.20,11.43,10.66,0.68,1.72,6.63,9.21,9.71,0.63,1.36,4.27
min,25.96,21.09,7.51,0.25,1.01,27.45,25.00,13.75,0.25,1.16,-11.00
25%,36.49,42.81,24.06,1.33,2.59,36.20,45.18,24.71,1.38,2.88,-2.00
50%,41.92,51.96,30.63,1.78,3.75,41.44,50.61,30.17,1.73,3.79,1.00
75%,46.96,57.90,38.87,2.24,4.94,46.21,56.45,38.77,2.21,4.69,2.00
max,61.62,71.68,66.67,3.92,10.29,58.65,71.64,66.67,3.67,8.08,16.00


In [8]:
# ── 2.5 Data Leakage 防呆驗證 ─────────────────────────────────
# 確認：滾動特徵與當場數據的相關性不應過高（若 > 0.95 則有洩漏嫌疑）
print("滾動特徵 vs 當場數據的相關係數（應明顯 < 1.0）：")
for col in GAME_STAT_COLS:
    corr = df_clean[col].corr(df_clean[f"{col}_roll3"])
    status = "OK" if abs(corr) < 0.95 else "WARNING: 可能存在洩漏"
    print(f"  {col} vs {col}_roll3: r = {corr:.3f}  [{status}]")

滾動特徵 vs 當場數據的相關係數（應明顯 < 1.0）：
  ASR vs ASR_roll3: r = 0.566  [OK]
  GP_pct vs GP_pct_roll3: r = 0.059  [OK]
  DIG_pct vs DIG_pct_roll3: r = 0.281  [OK]
  BLK_per_set vs BLK_per_set_roll3: r = 0.356  [OK]
  ACE_pct vs ACE_pct_roll3: r = 0.079  [OK]


---
## 3. 任務 2：時間序列交叉驗證 (TimeSeriesSplit)

### 為什麼不能用 StratifiedKFold？

| 方法 | 問題 |
|------|------|
| `StratifiedKFold` | 隨機抽取，訓練集可能包含 2026 年 1 月的比賽，測試集包含 2025 年 11 月 → 用未來預測過去 |
| `TimeSeriesSplit` | 嚴格依時間順序切分，訓練集永遠在測試集之前 → 模擬真實的「賽前預測」情境 |

```
Fold 1: Train [Nov] → Test [Dec 上旬]
Fold 2: Train [Nov ~ Dec 上旬] → Test [Dec 下旬]
Fold 3: Train [Nov ~ Dec 下旬] → Test [Jan 上旬]
...
```

In [9]:
# ── 3.1 建立特徵矩陣（全部使用賽前可得的滾動特徵） ──────────
# 按時間排序（跨隊），這是 TimeSeriesSplit 的前提
df_ts = df_clean.sort_values(["match_date", "team_id"]).reset_index(drop=True)

X_all = df_ts[ROLLING_FEATURES].values
y_all = df_ts["win"].values

print(f"特徵矩陣 shape: {X_all.shape}")
print(f"特徵欄位 ({len(ROLLING_FEATURES)})：")
for i, name in enumerate(ROLLING_FEATURES):
    print(f"  [{i:2d}] {name}")

特徵矩陣 shape: (203, 11)
特徵欄位 (11)：
  [ 0] ASR_roll3
  [ 1] GP_pct_roll3
  [ 2] DIG_pct_roll3
  [ 3] BLK_per_set_roll3
  [ 4] ACE_pct_roll3
  [ 5] ASR_roll5
  [ 6] GP_pct_roll5
  [ 7] DIG_pct_roll5
  [ 8] BLK_per_set_roll5
  [ 9] ACE_pct_roll5
  [10] win_streak


In [10]:
# ── 3.2 TimeSeriesSplit 切分 & 各 Fold 時間跨度 ───────────────
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

print(f"TimeSeriesSplit (n_splits={N_SPLITS})")
print("=" * 75)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all), 1):
    train_dates = df_ts.iloc[train_idx]["match_date"]
    test_dates = df_ts.iloc[test_idx]["match_date"]
    train_wins = y_all[train_idx].sum()
    test_wins = y_all[test_idx].sum()

    print(
        f"Fold {fold}: "
        f"Train [{train_dates.min()} ~ {train_dates.max()}] "
        f"{len(train_idx):3d} 筆 (勝{train_wins:3d}) │ "
        f"Test [{test_dates.min()} ~ {test_dates.max()}] "
        f"{len(test_idx):3d} 筆 (勝{test_wins:3d})"
    )

print("=" * 75)

TimeSeriesSplit (n_splits=5)
Fold 1: Train [2025-11-02 ~ 2025-11-23]  38 筆 (勝 20) │ Test [2025-11-23 ~ 2025-12-14]  33 筆 (勝 16)
Fold 2: Train [2025-11-02 ~ 2025-12-14]  71 筆 (勝 36) │ Test [2025-12-14 ~ 2026-01-03]  33 筆 (勝 17)
Fold 3: Train [2025-11-02 ~ 2026-01-03] 104 筆 (勝 53) │ Test [2026-01-03 ~ 2026-01-18]  33 筆 (勝 16)
Fold 4: Train [2025-11-02 ~ 2026-01-18] 137 筆 (勝 69) │ Test [2026-01-24 ~ 2026-02-08]  33 筆 (勝 17)
Fold 5: Train [2025-11-02 ~ 2026-02-08] 170 筆 (勝 86) │ Test [2026-02-08 ~ 2026-03-15]  33 筆 (勝 16)


In [11]:
# ── 3.3 Baseline：Random Forest + TimeSeriesSplit ─────────────
rf_baseline = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=3,
    random_state=SEED, class_weight="balanced",
)

rf_fold_scores = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all), 1):
    rf_baseline.fit(X_all[train_idx], y_all[train_idx])
    pred = rf_baseline.predict(X_all[test_idx])
    f1 = f1_score(y_all[test_idx], pred)
    rf_fold_scores.append(f1)
    print(f"  Fold {fold}: F1 = {f1:.3f}")

print(f"\nRandom Forest Baseline (TimeSeriesSplit):")
print(f"  Mean F1 = {np.mean(rf_fold_scores):.3f} (+/- {np.std(rf_fold_scores):.3f})")

  Fold 1: F1 = 0.811
  Fold 2: F1 = 0.667
  Fold 3: F1 = 0.824
  Fold 4: F1 = 0.645
  Fold 5: F1 = 0.625

Random Forest Baseline (TimeSeriesSplit):
  Mean F1 = 0.714 (+/- 0.085)


---
## 4. 任務 3：Optuna 自動調參 (XGBoost)

### 搜索空間

| 參數 | 範圍 | 說明 |
|------|------|------|
| `max_depth` | 2 ~ 8 | 樹的最大深度，過深容易 overfit |
| `learning_rate` | 0.01 ~ 0.3 | 學習率，越小需越多 estimators |
| `n_estimators` | 50 ~ 500 | 弱學習器數量 |
| `subsample` | 0.5 ~ 1.0 | 每棵樹的樣本抽取比例 |
| `colsample_bytree` | 0.5 ~ 1.0 | 每棵樹的特徵抽取比例 |
| `min_child_weight` | 1 ~ 10 | 葉節點最小樣本權重 |

In [12]:
# ── 4.1 定義 Optuna Objective 函數 ────────────────────────────
# 驗證策略：使用 TimeSeriesSplit，確保時間一致性

def objective(trial: optuna.Trial) -> float:
    """Optuna 目標函數：最大化 TimeSeriesSplit 下的平均 F1-score。"""

    params = {
        "max_depth":        trial.suggest_int("max_depth", 2, 8),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators":     trial.suggest_int("n_estimators", 50, 500, step=50),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        # 固定參數
        "random_state": SEED,
        "eval_metric": "logloss",
    }

    model = xgb.XGBClassifier(**params)

    fold_scores = []
    for train_idx, test_idx in tscv.split(X_all):
        model.fit(X_all[train_idx], y_all[train_idx])
        pred = model.predict(X_all[test_idx])
        fold_scores.append(f1_score(y_all[test_idx], pred))

    return np.mean(fold_scores)


print("Objective 函數定義完成。")

Objective 函數定義完成。


In [13]:
# ── 4.2 執行 Optuna 自動搜索 ──────────────────────────────────
study = optuna.create_study(direction="maximize", study_name="tvl_xgb_tuning")
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n完成 {len(study.trials)} 次試驗")
print(f"最佳 F1-score: {study.best_value:.4f}")
print(f"\n最佳超參數：")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/100 [00:00<?, ?it/s]


完成 100 次試驗
最佳 F1-score: 0.7149

最佳超參數：
  max_depth: 3
  learning_rate: 0.016883113024313504
  n_estimators: 450
  subsample: 0.8411335818516916
  colsample_bytree: 0.528586678872999
  min_child_weight: 2


In [ ]:
# ── 4.3 Optuna 調參過程視覺化 ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：F1 收斂曲線
trials = study.trials
trial_nums = [t.number for t in trials]
trial_vals = [t.value for t in trials]
best_so_far = np.maximum.accumulate(trial_vals)

axes[0].scatter(trial_nums, trial_vals, alpha=0.3, s=15, label="各次試驗")
axes[0].plot(trial_nums, best_so_far, color="red", linewidth=2, label="最佳 F1")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("F1-score")
axes[0].set_title("Optuna 搜索收斂曲線")
axes[0].legend()

# 右圖：各參數重要性
param_importance = optuna.importance.get_param_importances(study)
params_sorted = sorted(param_importance.items(), key=lambda x: x[1])
axes[1].barh(
    [p[0] for p in params_sorted],
    [p[1] for p in params_sorted],
    color="coral",
)
axes[1].set_xlabel("Importance")
axes[1].set_title("超參數重要性排名")

plt.tight_layout()
plt.show()

---
## 5. 最終模型訓練與評估

使用 Optuna 找到的最佳超參數，重新訓練最終模型。
以最後一個 TimeSeriesSplit fold 作為最終測試集，模擬真實預測場景。

In [ ]:
# ── 5.1 用最佳超參數訓練最終模型 ──────────────────────────────
best_params = study.best_params.copy()
best_params["random_state"] = SEED
best_params["eval_metric"] = "logloss"

final_model = xgb.XGBClassifier(**best_params)

# 最終評估：使用最後一個 fold 的 train/test split
splits = list(tscv.split(X_all))
final_train_idx, final_test_idx = splits[-1]

final_model.fit(X_all[final_train_idx], y_all[final_train_idx])
final_pred = final_model.predict(X_all[final_test_idx])
final_proba = final_model.predict_proba(X_all[final_test_idx])[:, 1]

print("=== 最終模型 (Optuna-tuned XGBoost + 滾動特徵 + TimeSeriesSplit) ===")
print(f"訓練集：{len(final_train_idx)} 筆 ({df_ts.iloc[final_train_idx]['match_date'].min()} ~ {df_ts.iloc[final_train_idx]['match_date'].max()})")
print(f"測試集：{len(final_test_idx)} 筆 ({df_ts.iloc[final_test_idx]['match_date'].min()} ~ {df_ts.iloc[final_test_idx]['match_date'].max()})")
print()
print(classification_report(y_all[final_test_idx], final_pred, target_names=["敗 (0)", "勝 (1)"]))

In [ ]:
# ── 5.2 混淆矩陣 ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_all[final_test_idx], final_pred,
    display_labels=["敗", "勝"],
    cmap="RdYlGn",
    ax=ax,
)
ax.set_title("最終模型 — 混淆矩陣 (TimeSeriesSplit 最後一折)")
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 與 v1 Baseline 比較 ───────────────────────────────────
# v1 的數字（從 02 notebook 讀取）
V1_CV_F1 = 0.717   # v1 XGBoost StratifiedKFold CV F1
V1_NOTE = "StratifiedKFold + 當場賽後數據"

comparison = pd.DataFrame([
    {
        "版本": "v1 (賽後解讀)",
        "驗證策略": V1_NOTE,
        "CV F1": V1_CV_F1,
        "備註": "有 Data Leakage，F1 偏高不可信",
    },
    {
        "版本": "v2 Baseline (RF)",
        "驗證策略": "TimeSeriesSplit + 滾動特徵",
        "CV F1": np.mean(rf_fold_scores),
        "備註": "Random Forest 預設參數",
    },
    {
        "版本": "v2 Final (XGB+Optuna)",
        "驗證策略": "TimeSeriesSplit + 滾動特徵",
        "CV F1": study.best_value,
        "備註": "Optuna 100 trials 最佳超參數",
    },
])
comparison["CV F1"] = comparison["CV F1"].round(3)

print("=== v1 vs v2 模型比較 ===")
print(comparison.to_string(index=False))
print()
print("注意：v1 的 F1 較高是因為 Data Leakage（用答案預測答案），")
print("v2 的 F1 才是真實的賽前預測能力指標。")

### 5.4 SHAP 特徵重要性分析

使用滾動特徵的 SHAP 分析結果才有實戰意義：
「過去 N 場的哪項表現最能預示下一場的勝負？」

In [ ]:
# ── 5.4a 用全部資料重新訓練最終模型（供 SHAP + 匯出） ────────
final_model_full = xgb.XGBClassifier(**best_params)
final_model_full.fit(X_all, y_all)

# SHAP
explainer = shap.TreeExplainer(final_model_full)
shap_values = explainer.shap_values(X_all)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# 中文特徵名稱
ROLLING_LABELS = {
    "ASR_roll3": "近3場 攻擊率", "ASR_roll5": "近5場 攻擊率",
    "GP_pct_roll3": "近3場 接發率", "GP_pct_roll5": "近5場 接發率",
    "DIG_pct_roll3": "近3場 防守率", "DIG_pct_roll5": "近5場 防守率",
    "BLK_per_set_roll3": "近3場 局均攔網", "BLK_per_set_roll5": "近5場 局均攔網",
    "ACE_pct_roll3": "近3場 發球率", "ACE_pct_roll5": "近5場 發球率",
    "win_streak": "連勝/連敗",
}
feature_labels = [ROLLING_LABELS.get(c, c) for c in ROLLING_FEATURES]

print(f"SHAP values shape: {shap_vals.shape}")

In [ ]:
# SHAP Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_vals, features=X_all,
    feature_names=feature_labels,
    show=False,
)
plt.title("v2 最終模型 — SHAP Summary Plot（賽前滾動特徵對勝場的貢獻）")
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot + Top 3 關鍵指標
plt.figure(figsize=(10, 5))
shap.summary_plot(
    shap_vals, features=X_all,
    feature_names=feature_labels,
    plot_type="bar",
    show=False,
)
plt.title("v2 最終模型 — SHAP 平均特徵貢獻度")
plt.tight_layout()
plt.show()

# Top 3
mean_abs = np.abs(shap_vals).mean(axis=0)
top3 = np.argsort(mean_abs)[::-1][:3]

print("=" * 60)
print("v2 賽前預測引擎 — 影響勝負的前三大關鍵歷史指標：")
print("=" * 60)
for rank, idx in enumerate(top3, 1):
    print(f"  {rank}. {feature_labels[idx]} ({ROLLING_FEATURES[idx]})")
    print(f"     平均 |SHAP|: {mean_abs[idx]:.4f}")
print("=" * 60)

---
## 6. 模型匯出

In [ ]:
MODEL_DIR = Path("../src/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "match_predictor.pkl"

artifact = {
    "model": final_model_full,
    "model_name": "XGBoost (Optuna-tuned, v2)",
    "version": "v2",
    "feature_cols": ROLLING_FEATURES,
    "feature_labels": feature_labels,
    "best_params": best_params,
    "optuna_best_f1": study.best_value,
    "training_samples": len(df_clean),
    "game_stat_cols": GAME_STAT_COLS,  # 計算滾動特徵時需要的原始欄位名
}

joblib.dump(artifact, model_path)

loaded = joblib.load(model_path)
print(f"模型已儲存至：{model_path.resolve()}")
print(f"版本：{loaded['version']}")
print(f"特徵數：{len(loaded['feature_cols'])}")
print(f"Optuna 最佳 F1: {loaded['optuna_best_f1']:.3f}")
print(f"檔案大小：{model_path.stat().st_size / 1024:.1f} KB")

---
## 7. 整合注意事項與結論

### 三個模組的整合重點

| 模組 | 關鍵防呆 |
|------|----------|
| **滾動特徵** | `shift(1)` 是核心 — 忘了就是 Data Leakage。同日比賽需穩定排序。首場紀錄無歷史需移除。 |
| **TimeSeriesSplit** | 資料必須先按時間排序。各 fold 的時間跨度應印出驗證，確認沒有「未來 → 過去」的切分。 |
| **Optuna** | Objective 函數內部必須用 `TimeSeriesSplit`（不是 StratifiedKFold），否則調出來的參數在時序驗證下不一定最優。 |

### 已知限制與風險

1. **Proxy Label 污染放大**：勝負標籤是用總得分推論的近似值，
   當這些標籤被用來計算 `win_streak` 滾動特徵時，標籤誤差會隨時間傳播。
   後續應優先引入外部 API 局比分建立真實標籤。

2. **同日排序不確定性**：`match_date` 僅有日期，同日雙重賽的場次先後順序
   目前以 `opponent` 名稱排序作為 tiebreaker，並非真實開打順序。

3. **樣本量瓶頸**：移除首場後僅約 200 筆樣本，11 個特徵，
   仍有過擬合風險。隨賽季進行應持續補充數據。

### 後續改進方向

- 加入「對手近期表現」差異特徵（如：我方近3場ASR - 對手近3場ASR）
- 引入外部 API 取得真實局比分，消除 proxy label 偏差
- 嘗試加入賽程因素（連續兩天出賽 = 疲勞度）
- 模型上線後追蹤 calibration（預測機率 vs 實際勝率）